# IS630: Green Light for an Indie Studio
## Notebook 02: Cleaning and Feature Engineering
**Group**: [Insert names]

> **AI Coding Assistant Declaration**:
> In accordance with course guidelines, an AI coding assistant was used strictly for code scaffolding and plotting functions. All code blocks below are labeled as AI-assisted. All modelling decisions, interpretations, and written commentaries are independently authored by group members in the designated sections below.

This notebook works through one data question per section, building towards **AQ2**


- Apply exclusions decided in 01: software, zero-review games, implausible playtime, free-to-play (decide)
- Story-driven flag, with the rule written down
- Build the target(s) chosen for AQ1
- Encode features: genres, price bands, skewed-variable transforms, game age
- Fix inclusion thresholds before any model is fitted
- Split a validation set out of train
- Save train, validation and the feature spec

*Outline written by AI Assistant*

In [1]:
# import libraries and load data
import sys
import re
import ast
from pathlib import Path

import numpy as np 
import pandas as pd
from IPython.display import display

# Locate the project root
ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "src" / "is630").exists()
)

sys.path.insert(0, str(ROOT / "src"))

from is630 import config

# Load the output from notebook 01
input_path = config.INTERIM / "01_eda.parquet"

if not input_path.exists():
    raise FileNotFoundError(
        "Run Notebook 01 and save 01_eda.parquet first"
    )

df = pd.read_parquet(input_path)
config.verify_appid_fingerprint(df, config.TRAIN_APPID_SHA256, "01_eda.parquet")

required_columns = [
    "steam_appid",
    "steam_is_software",
    "steam_positive",
    "steam_negative",
    "steam_review_count",
    "steam_price",
    "steam_playtime_hours",
    "matched_hltb",
    "hltb_main_story",
]

missing = set(required_columns) - set(df.columns)
assert not missing, f"Missing required columns: {missing}"

print(f"Dataset loaded: {len(df):,} rows")
print(f"Number of columns: {df.shape[1]}")
print(f"Unique games: {df['steam_appid'].nunique():,}")

display(df.head())

Dataset loaded: 5,281 rows
Number of columns: 85
Unique games: 5,281


,steam_appid,steam_name,steam_release_date,steam_estimated_owners,steam_peak_ccu,steam_required_age,steam_price,steam_discount,steam_dlc_count,steam_about_the_game,...,steam_positive_rate,steam_playtime_hours,steam_years_since_release,steam_language_count,steam_has_metacritic,completion_ratio,steam_owners_lower,steam_owners_upper,steam_is_software,steam_review_bucket
0,2358720,Black Myth: Wukong,"Aug 19, 2024",50000000 - 100000000,15004,13,59.99,0,2,Black Myth: Wukong is an action RPG rooted in ...,...,0.966631,46.866667,2,15,False,1.228162,50000000,100000000,False,20k+
1,1091500,Cyberpunk 2077,"Dec 9, 2020",20000000 - 50000000,22265,17,20.99,65,3,None,...,0.843950,53.550000,6,19,True,2.058824,20000000,50000000,False,20k+
2,292030,The Witcher 3: Wild Hunt,"May 18, 2015",20000000 - 50000000,15893,17,3.99,90,22,None,...,0.961266,14.583333,11,17,True,0.282623,20000000,50000000,False,20k+
3,1086940,Baldur's Gate 3,"Aug 3, 2023",20000000 - 50000000,54771,17,44.99,25,2,Gather your party and return to the Forgotten ...,...,0.968483,78.300000,3,15,True,1.075106,20000000,50000000,False,20k+
4,1938090,Call of Duty®,"Oct 27, 2022",50000000 - 100000000,67419,17,0.00,45,43,The Call of Duty® experience supports Call of ...,...,0.587573,11.950000,4,14,False,1.741983,50000000,100000000,False,20k+


### 1.1 Initial Data Quality Assessment

An initial data quality assessment was conducted to identify observations requiring exclusion or special treatment before statistical modelling.

Of the 5,281 observations in the training dataset, 0 (0.00%) were classified as non-game software and were identified for exclusion.

A total of 15 observations (0.28%) had zero Steam reviews, making their positive review rates undefined. These observations will be retained in the cleaned base dataset but excluded from analyses requiring an observed review proportion.

Furthermore, 1,676 observations (31.74%) had zero recorded Steam playtime. Since these observations cannot provide a meaningful positive completion ratio, they will be excluded from completion-ratio modelling while remaining available for other analyses.

Only 3 games had median playtime exceeding 1,000 hours, and none exceeded 8,760 hours. Extreme observations will be flagged and evaluated using a predefined threshold.

Free-to-play games accounted for 7.04% of observations and will be retained to preserve their representation in the Steam market.

No negative or missing price values were identified.

These decisions establish a consistent approach to data cleaning while minimizing unnecessary information loss and preserving observations for different analytical objectives.

In [2]:
# Inspect candidate exclusions before removing anything

review_count = pd.to_numeric(
    df["steam_review_count"], errors="coerce"
)

playtime = pd.to_numeric(
    df["steam_playtime_hours"], errors="coerce"
)

price = pd.to_numeric(
    df["steam_price"], errors="coerce"
)

audit = pd.DataFrame({
    "Condition": [
        "Non-game software",
        "Zero-review games",
        "Zero Steam playtime",
        "Median playtime > 1,000 hours",
        "Median playtime > 8,760 hours",
        "Free-to-play games",
        "Negative prices",
        "Missing prices"
    ],
    "Number of games": [
        df["steam_is_software"].fillna(False).sum(),
        review_count.eq(0).sum(),
        playtime.eq(0).sum(),
        playtime.gt(1000).sum(),
        playtime.gt(8760).sum(),
        price.eq(0).sum(),
        price.lt(0).sum(),
        price.isna().sum()
    ]
})

audit["Percentage (%)"] = (
    audit["Number of games"] / len(df) * 100
).round(2)

display(audit)


,Condition,Number of games,Percentage (%)
0,Non-game software,0,0.00
1,Zero-review games,15,0.28
2,Zero Steam playtime,1676,31.74
3,"Median playtime > 1,000 hours",3,0.06
4,"Median playtime > 8,760 hours",0,0.00
5,Free-to-play games,372,7.04
6,Negative prices,0,0.00
7,Missing prices,0,0.00


In [3]:

# Identify invalid observations and add eligibility flags
# The steps live in src/is630/features.py, so notebook 07 applies them to the test holdout unchanged.
from is630.features import valid_rows, add_cleaning_flags

software = df["steam_is_software"].fillna(False)
exclude = ~valid_rows(df)    # software, missing or negative review counts, negative price
df_clean = add_cleaning_flags(df.loc[~exclude])

print(f"Original games: {len(df):,}")
print(f"Excluded games: {exclude.sum():,}")
print(f"Retained games: {len(df_clean):,}")

print(
    "Retained games with reviews:",
    f"{(~df_clean['flag_zero_reviews']).sum():,}"
)

print(
    "Retained free-to-play games:",
    f"{df_clean['flag_free_to_play'].sum():,}"
)

print(
    "Usable Steam playtime observations:",
    f"{df_clean['steam_playtime_hours_usable'].notna().sum():,}"
)

assert df_clean["steam_appid"].is_unique


Original games: 5,281
Excluded games: 0
Retained games: 5,281
Retained games with reviews: 5,266
Retained free-to-play games: 372
Usable Steam playtime observations: 3,605


In [4]:

cleaning_summary = pd.DataFrame({
    "Stage": [
        "Initial dataset",
        "Non-game software removed",
        "Invalid records removed (including overlaps)",
        "Cleaned base dataset"
    ],
    "Games": [
        len(df),
        int(software.sum()),
        int((exclude & ~software).sum()),
        len(df_clean)
    ]
})

cleaning_summary["Percentage of original"] = (
    cleaning_summary["Games"] / len(df) * 100
).round(2)

display(cleaning_summary)

assert len(df_clean) + int(exclude.sum()) == len(df)


,Stage,Games,Percentage of original
0,Initial dataset,5281,100.0
1,Non-game software removed,0,0.0
2,Invalid records removed (including overlaps),0,0.0
3,Cleaned base dataset,5281,100.0


In [5]:
# Step 2: Inspect metadata used for story-driven classification

metadata_cols = [
    "steam_name",
    "steam_genres",
    "steam_categories",
    "steam_tags"
]

display(df_clean[metadata_cols].sample(
    10, random_state=config.SEED
))

print("Missing metadata values:")
print(
    df_clean[metadata_cols]
    .isna()
    .sum()
)

print("\nSingle-player games:")
print(
    df_clean["steam_categories"]
    .fillna("")
    .str.contains("Single-player", case=False, regex=False)
    .sum()
)


,steam_name,steam_genres,steam_categories,steam_tags
8,Hogwarts Legacy,"Action,Adventure,RPG","Single-player,Steam Achievements,Full controll...","Magic,Open World,Fantasy,Singleplayer,Adventur..."
1580,Dreamfall: The Longest Journey,"Adventure,RPG","Single-player,Family Sharing","Adventure,Female Protagonist,Story Rich,Fantas..."
544,The Dungeon Of Naheulbeuk: The Amulet Of Chaos,"Action,Adventure,Indie,RPG,Strategy","Single-player,Steam Achievements,Full controll...","RPG,Turn-Based Tactics,Adventure,Turn-Based,Si..."
5201,Romance MD: Always On Call,Adventure,"Single-player,Partial Controller Support,Famil...","Visual Novel,Adventure,Drama,Otome,Choose Your..."
3942,Pendragon Rising,"Indie,RPG","Single-player,Steam Achievements,Captions avai...","RPG,Indie,Text-Based,Choose Your Own Adventure..."
2312,Umineko: Golden Fantasia,"Action,Indie","Single-player,Multi-player,PvP,Online PvP,Shar...","Action,Fighting,2D Fighter,Indie,Anime,Multipl..."
3407,Blackberry,"Action,Adventure,Casual,Indie","Single-player,Steam Achievements,Steam Leaderb...","Adventure,Action,Indie,Horror,Casual,Survival ..."
1503,Locked Up,Indie,"Single-player,Steam Achievements,Partial Contr...","Indie,Violent,Horror,Psychological Horror,Firs..."
167,Company of Heroes 3,"Action,Strategy","Single-player,Multi-player,PvP,Online PvP,Co-o...","Singleplayer,Strategy,Wargame,RTS,Multiplayer,..."
346,Deus Ex: Game of the Year Edition,Action,"Single-player,Multi-player,Full controller sup...","Cyberpunk,RPG,Stealth,Classic,Immersive Sim,FP..."


Missing metadata values:
steam_name           0
steam_genres        12
steam_categories     0
steam_tags           0
dtype: int64

Single-player games:
5281


### 1.2 Story-Driven Game Classification
*Written by AI Assistant, revised for the current scope. To be reviewed by the group.*

The story-driven rule is applied in notebook 00, before the train and test split, by `classify_story_driven` in `src/is630/features.py`. It uses Steam categories, tags and genres only. A game is story-driven if it supports single-player play and carries a narrative tag such as `Story Rich`, `Visual Novel`, `Interactive Fiction` or `Choices Matter`.

This section re-applies the rule to the training data as a check. All 5,281 games are single-player and story-driven, and the re-applied rule matches the stored flag for every game. All 5,281 are matched to HowLongToBeat with a positive main-story time, because notebook 00 keeps matched games only.

No game has missing category or tag information. Games with no tags or categories cannot be classified, so they left the scope in notebook 00. All validation checks passed.

In [6]:
# Story-driven flags: single-player, narrative tags, Visual Novel genre, and the final rule
# (STORY_TAGS and classify_story_driven). Also flags missing category or tag metadata.
# The steps live in src/is630/features.py, so notebook 07 applies them to the test holdout unchanged.
from is630.features import add_story_flags

df_clean = add_story_flags(df_clean)

print("Story-driven classification completed.")
print(
    f"Story-driven games: "
    f"{df_clean['flag_story_driven'].sum():,}"
)

print(
    f"Percentage of cleaned dataset: "
    f"{df_clean['flag_story_driven'].mean() * 100:.2f}%"
)


Story-driven classification completed.
Story-driven games: 5,281
Percentage of cleaned dataset: 100.00%


In [7]:

import sys
import importlib
from pathlib import Path

# Locate the shared features module
current = Path.cwd().resolve()

project_root = next(
    (
        folder for folder in [current, *current.parents]
        if (folder / "src" / "is630" / "features.py").exists()
    ),
    None
)

assert project_root is not None, "Project root not found"

sys.path.insert(0, str(project_root / "src"))

import is630.features as features
importlib.reload(features)

# Compare original and refactored classifications
original = df_clean["flag_story_driven"]
refactored = features.classify_story_driven(df_clean)

print("Original story-driven count:", original.sum())
print("Refactored story-driven count:", refactored.sum())
print("Mismatched classifications:", (original != refactored).sum())

assert original.equals(refactored)

print("All classifications match!")


Original story-driven count: 5281
Refactored story-driven count: 5281
Mismatched classifications: 0
All classifications match!


In [8]:

story_summary = pd.DataFrame({
    "Condition": [
        "All cleaned games",
        "Single-player games",
        "Games with narrative tags",
        "Games with Visual Novel genre",
        "Flagged story-driven games",
        "Story-driven and HLTB matched",
        "Story-driven with positive HLTB main-story time"
    ],
    "Number of games": [
        len(df_clean),
        int(df_clean["flag_single_player"].sum()),
        int(df_clean["flag_narrative_tags"].sum()),
        int(df_clean["flag_visual_novel_genre"].sum()),
        int(df_clean["flag_story_driven"].sum()),
        int((
            df_clean["flag_story_driven"]
            & df_clean["matched_hltb"].fillna(False)
        ).sum()),
        int((
            df_clean["flag_story_driven"]
            & (df_clean["hltb_main_story"] > 0)
        ).sum())
    ]
})

story_summary["Percentage of cleaned (%)"] = (
    story_summary["Number of games"] / len(df_clean) * 100
).round(2)

display(story_summary)

# Inspect example classifications
display(
    df_clean.loc[
        df_clean["flag_story_driven"],
        [
            "steam_name",
            "steam_genres",
            "steam_tags",
            "steam_review_count",
            "matched_hltb"
        ]
    ].sample(
        min(10, int(df_clean["flag_story_driven"].sum())),
        random_state=config.SEED
    )
)


,Condition,Number of games,Percentage of cleaned (%)
0,All cleaned games,5281,100.0
1,Single-player games,5281,100.0
2,Games with narrative tags,5281,100.0
3,Games with Visual Novel genre,0,0.0
4,Flagged story-driven games,5281,100.0
5,Story-driven and HLTB matched,5281,100.0
6,Story-driven with positive HLTB main-story time,5281,100.0


,steam_name,steam_genres,steam_tags,steam_review_count,matched_hltb
8,Hogwarts Legacy,"Action,Adventure,RPG","Magic,Open World,Fantasy,Singleplayer,Adventur...",321395,True
1580,Dreamfall: The Longest Journey,"Adventure,RPG","Adventure,Female Protagonist,Story Rich,Fantas...",1152,True
544,The Dungeon Of Naheulbeuk: The Amulet Of Chaos,"Action,Adventure,Indie,RPG,Strategy","RPG,Turn-Based Tactics,Adventure,Turn-Based,Si...",7469,True
5201,Romance MD: Always On Call,Adventure,"Visual Novel,Adventure,Drama,Otome,Choose Your...",5,True
3942,Pendragon Rising,"Indie,RPG","RPG,Indie,Text-Based,Choose Your Own Adventure...",96,True
2312,Umineko: Golden Fantasia,"Action,Indie","Action,Fighting,2D Fighter,Indie,Anime,Multipl...",478,True
3407,Blackberry,"Action,Adventure,Casual,Indie","Adventure,Action,Indie,Horror,Casual,Survival ...",164,True
1503,Locked Up,Indie,"Indie,Violent,Horror,Psychological Horror,Firs...",1301,True
167,Company of Heroes 3,"Action,Strategy","Singleplayer,Strategy,Wargame,RTS,Multiplayer,...",33246,True
346,Deus Ex: Game of the Year Edition,Action,"Cyberpunk,RPG,Stealth,Classic,Immersive Sim,FP...",14552,True


In [9]:

# Record missing metadata that may affect classification

# flag_missing_story_metadata is added by add_story_flags above

print(
    "Games with incomplete story metadata:",
    f"{df_clean['flag_missing_story_metadata'].sum():,}"
)

print(
    "Story-driven games with incomplete metadata:",
    f"{(df_clean['flag_story_driven'] & df_clean['flag_missing_story_metadata']).sum():,}"
)

# Verify logical consistency
assert not (
    df_clean["flag_story_driven"]
    & ~df_clean["flag_single_player"]
).any()

assert (
    df_clean["flag_story_driven"].sum() > 0
)

print("Story-driven classification checks passed.")


Games with incomplete story metadata: 0
Story-driven games with incomplete metadata: 0
Story-driven classification checks passed.


### 1.3 Target Variable Construction

Three target variables were constructed to support subsequent statistical modelling: Steam review count, positive review rate, and completion ratio.

Review count was calculated as the sum of positive and negative reviews and was available for all 5,281 games. The median review count was 333, compared to a mean of 5,600.52, indicating a strongly right-skewed distribution.

Positive review rate was calculated as the proportion of positive reviews among total reviews. This was available for 5,266 games (99.72%), with a median of 0.870. Games with zero reviews were assigned missing values because their review proportions were undefined.

Completion ratio was calculated by dividing median Steam playtime by HLTB main-story duration for eligible story-driven games. A total of 3,605 games (68.26%) satisfied these requirements, with a median completion ratio of 0.822 and a mean of 1.794.

Eligibility flags were created for subsequent modelling, and validation checks confirmed that all target variables satisfied their expected value constraints.

The final modelling thresholds will be established before model fitting to ensure consistent sample selection.

In [10]:
# Step 3: Construct target variables
# Review count, positive rate and completion ratio come from notebook 01 (features.add_derived).
# This step adds each target's eligibility flag.
# The steps live in src/is630/features.py, so notebook 07 applies them to the test holdout unchanged.
from is630.features import add_target_flags

df_clean = add_target_flags(df_clean)

print("Target variables successfully constructed.")


Target variables successfully constructed.


In [11]:

# Summarize target availability

target_cols = [
    "steam_review_count",
    "steam_positive_rate",
    "completion_ratio"
]

target_summary = pd.DataFrame({
    "Target": target_cols,
    "Available games": [
        df_clean[col].notna().sum()
        for col in target_cols
    ],
    "Missing games": [
        df_clean[col].isna().sum()
        for col in target_cols
    ],
    "Median": [
        df_clean[col].median()
        for col in target_cols
    ],
    "Mean": [
        df_clean[col].mean()
        for col in target_cols
    ]
})

target_summary["Coverage (%)"] = (
    target_summary["Available games"]
    / len(df_clean) * 100
).round(2)

display(target_summary.round(3))

# Check target validity
assert df_clean["steam_review_count"].notna().all()
assert df_clean["steam_review_count"].ge(0).all()

assert df_clean["steam_positive_rate"].dropna().between(
    0, 1
).all()

assert (
    df_clean.loc[
        df_clean["flag_completion_eligible"],
        "completion_ratio"
    ].gt(0).all()
)



# Verify review-rate availability
assert (
    df_clean["steam_positive_rate"].notna()
    == df_clean["flag_reception_eligible"]
).all()

# Verify completion-ratio availability
assert (
    df_clean.loc[
        df_clean["flag_completion_eligible"],
        "completion_ratio"
    ].notna().all()
)

print("All target validation checks passed.")



,Target,Available games,Missing games,Median,Mean,Coverage (%)
0,steam_review_count,5281,0,333.000,5600.515,100.00
1,steam_positive_rate,5266,15,0.870,0.834,99.72
2,completion_ratio,5281,0,0.423,1.225,100.00


All target validation checks passed.


### 1.4 Feature Engineering and Encoding
*Written by AI Assistant, revised for the current scope. To be reviewed by the group.*

Eight Steam genres were converted into binary indicators: Indie, Casual, Action, Adventure, Simulation, Strategy, RPG and Early Access. A game can belong to several genres. A separate indicator marks the 12 games (0.23%) with no genre information.

Price was transformed with `log1p()` and grouped into five price bands: Free, $0-5, $5-10, $10-20 and $20+. Game age is kept as a numeric predictor, from 1 to 28 years.

The language count was checked after 8 games showed 103 languages. Their source strings were truncated and could not be parsed. A revised parsing method processed all 5,281 records and found 8 one-count discrepancies against the original calculation. These corrections were kept, although some uncertainty remains due to the use of fallback parsing.

No game has zero or missing languages, so all 5,281 games have a usable language count, transformed with `log1p()`. A median fill is still set up in section 1.6, computed on the development-training split only, but it has nothing to fill in this sample.

Validation checks confirmed that genre indicators are binary, every game has a price band, and the numeric features fall within their expected ranges.

In [12]:
# Step 4: Inspect candidate predictor variables

numeric_features = [
    "steam_price",
    "steam_years_since_release",
    "steam_language_count"
]

# Check that the expected features are available
missing = set(numeric_features + ["steam_genres"]) - set(df_clean.columns)
assert not missing, f"Missing features: {missing}"

print("Numeric feature summary:")
display(
    df_clean[numeric_features].describe(
        percentiles=[0.25, 0.50, 0.75, 0.95, 0.99]
    ).T.round(2)
)

print("\nMissing values:")
display(df_clean[numeric_features + ["steam_genres"]].isna().sum())

# Examine the most common game genres
genre_counts = (
    df_clean["steam_genres"]
    .dropna()
    .str.split(",")
    .explode()
    .str.strip()
    .value_counts()
)

print("\nTop 15 genres:")
display(genre_counts.head(15))


Numeric feature summary:


,count,mean,std,min,25%,50%,75%,95%,99%,max
steam_price,5281.0,6.46,10.31,0.0,1.99,4.39,8.74,19.94,29.99,500.0
steam_years_since_release,5281.0,6.22,3.69,1.0,3.00,6.00,9.00,12.00,17.00,28.0
steam_language_count,5281.0,5.64,6.41,1.0,1.00,4.00,9.00,15.00,27.00,103.0



Missing values:


steam_price                   0
steam_years_since_release     0
steam_language_count          0
steam_genres                 12
dtype: int64


Top 15 genres:


steam_genres
Adventure                3570
Indie                    3558
Casual                   1620
Action                   1485
RPG                      1376
Simulation                907
Strategy                  582
Free To Play              234
Early Access               57
Sports                     46
Racing                     25
Massively Multiplayer      12
Nudity                      9
Sexual Content              7
Violent                     6
Name: count, dtype: int64

In [13]:
# Investigate unusual language counts

lang = df_clean["steam_language_count"]

print("Games with 103 languages:", lang.eq(103).sum())
print("Games with 0 languages:", lang.eq(0).sum())
print("Games with more than 30 languages:", lang.gt(30).sum())

print("\nMost common language counts:")
display(lang.value_counts().head(15))

# Examine the original source values
extreme_languages = df_clean.loc[
    lang.eq(103),
    [
        "steam_name",
        "steam_language_count",
        "steam_supported_languages"
    ]
]

print("\nExamples of games with 103 languages:")

for _, row in extreme_languages.head(5).iterrows():
    print("\nGame:", row["steam_name"])
    print("Language count:", row["steam_language_count"])
    print(
        "Original value:",
        repr(row["steam_supported_languages"])[:600]
    )


Games with 103 languages: 8
Games with 0 languages: 0
Games with more than 30 languages: 13

Most common language counts:


steam_language_count
1     1587
2      635
3      352
5      319
4      309
9      273
6      269
8      257
10     199
7      198
11     192
12     172
13     129
14     122
15      71
Name: count, dtype: int64


Examples of games with 103 languages:

Game: Zup! X
Language count: 103
Original value: "['English', 'French', 'Italian', 'German', 'Spanish - Spain', 'Arabic', 'Bulgarian', 'Portuguese - Brazil', 'Hungarian', 'Greek', 'Danish', 'Traditional Chinese', 'Simplified Chinese', 'Korean', 'Dutch', 'Norwegian', 'Polish', 'Portuguese - Portugal', 'Romanian', 'Russian', 'Thai', 'Turkish', 'Ukrainian', 'Finnish', 'Czech', 'Swedish', 'Japanese', 'Vietnamese', 'Spanish - Latin America', 'Azerbaijani', 'Albanian', 'Amharic', 'Armenian', 'Assamese', 'Afrikaans', 'Basque', 'Belarusian', 'Bangla', 'Bosnian', 'Valencian', 'Welsh', 'Wolof', 'Galician', 'Georgian', 'Gujarati', 'Dari', 'Zulu', 'Hebr

Game: Papetura
Language count: 103
Original value: "['English', 'French', 'German', 'Spanish - Spain', 'Japanese', 'Korean', 'Polish', 'Portuguese - Brazil', 'Russian', 'Simplified Chinese', 'Traditional Chinese', 'Italian', 'Arabic', 'Bulgarian', 'Czech', 'Danish', 'Dutch', 'Finnish', 'Greek', 'Hungarian', 

In [14]:
# Step 4: Feature engineering and encoding
# Genres (GENRE_MAP), price and price bands (PRICE_BINS, PRICE_LABELS), Q4 release, platforms,
# audio languages, achievements, age rating, store categories (CATEGORY_FLAGS), missing tags,
# self-publishing, website and support contact, HLTB scope features.
# The steps live in src/is630/features.py, so notebook 07 applies them to the test holdout unchanged.
from is630.features import GENRE_MAP, PRICE_BINS, PRICE_LABELS, encode_features

df_clean = encode_features(df_clean)

print("Feature engineering completed.")


Feature engineering completed.

In [15]:

print("Unique column names:", df_clean.columns.is_unique)

print(
    "Price-band dummy columns:",
    df_clean.columns.str.startswith("price_band_").sum()
)


Unique column names: True
Price-band dummy columns: 5


In [16]:

# Verify feature engineering results

feature_cols = (
    list(GENRE_MAP.values())
    + ["flag_missing_genre"]
    + ["feature_log_price", "steam_years_since_release"]
)

print("Engineered feature summary:")
display(df_clean[feature_cols].describe().T.round(3))

print("\nPrice band distribution:")
display(
    df_clean["feature_price_band"]
    .value_counts()
    .sort_index()
)

print("\nMissing values in engineered features:")
display(df_clean[feature_cols].isna().sum())

# Validation checks
assert df_clean["feature_log_price"].ge(0).all()
assert df_clean["steam_years_since_release"].ge(0).all()


assert df_clean["feature_price_band"].notna().all()

assert (
    df_clean[list(GENRE_MAP.values())]
    .isin([0, 1])
    .all()
    .all()
)

assert (
    df_clean.filter(like="price_band_").sum(axis=1) == 1
).all()

print("\nAll feature encoding checks passed.")


Engineered feature summary:


,count,mean,std,min,25%,50%,75%,max
genre_indie,5281.0,0.674,0.469,0.0,0.000,1.000,1.000,1.000
genre_casual,5281.0,0.307,0.461,0.0,0.000,0.000,1.000,1.000
genre_action,5281.0,0.281,0.450,0.0,0.000,0.000,1.000,1.000
genre_adventure,5281.0,0.676,0.468,0.0,0.000,1.000,1.000,1.000
genre_simulation,5281.0,0.172,0.377,0.0,0.000,0.000,0.000,1.000
genre_strategy,5281.0,0.110,0.313,0.0,0.000,0.000,0.000,1.000
genre_rpg,5281.0,0.261,0.439,0.0,0.000,0.000,1.000,1.000
genre_early_access,5281.0,0.011,0.103,0.0,0.000,0.000,0.000,1.000
flag_missing_genre,5281.0,0.002,0.048,0.0,0.000,0.000,0.000,1.000
feature_log_price,5281.0,1.648,0.846,0.0,1.095,1.685,2.276,6.217



Price band distribution:


feature_price_band
Free      372
0-5      2773
5-10     1219
10-20     739
20+       178
Name: count, dtype: int64


Missing values in engineered features:


genre_indie                  0
genre_casual                 0
genre_action                 0
genre_adventure              0
genre_simulation             0
genre_strategy               0
genre_rpg                    0
genre_early_access           0
flag_missing_genre           0
feature_log_price            0
steam_years_since_release    0
dtype: int64


All feature encoding checks passed.


In [17]:
# Parse the original language lists more reliably
# features.add_language_features adds every language column used in this section:
# a strict parse (feature_language_count_parsed), the corrected count with a quote-splitting
# fallback (feature_language_count, language_parse_status), and the model-ready count.
# The steps live in src/is630/features.py, so notebook 07 applies them to the test holdout unchanged.
from is630.features import add_language_features

df_clean = add_language_features(df_clean)

# Compare original and parsed language counts
valid = df_clean["feature_language_count_parsed"].notna()

mismatches = (
    df_clean.loc[valid, "steam_language_count"]
    != df_clean.loc[valid, "feature_language_count_parsed"]
).sum()

print("Successfully parsed games:", valid.sum())
print("Missing or unparseable entries:", (~valid).sum())
print("Count mismatches:", mismatches)

print("\nGames originally recorded with 103 languages:")

display(
    df_clean.loc[
        df_clean["steam_language_count"] == 103,
        [
            "steam_name",
            "steam_language_count",
            "feature_language_count_parsed"
        ]
    ].head(10)
)


Successfully parsed games: 5273
Missing or unparseable entries: 8
Count mismatches: 0

Games originally recorded with 103 languages:


,steam_name,steam_language_count,feature_language_count_parsed
643,Zup! X,103,NaN
1753,Papetura,103,NaN
2520,Last Room,103,NaN
2997,Shapik: The Moon Quest,103,NaN
3842,Winter,103,NaN
4590,Quadrata,103,NaN
4972,One Chance,103,NaN
5087,Tiny Onion Knight,103,NaN


In [18]:
# Investigate records that failed language parsing
examples = df_clean.loc[
    df_clean["steam_language_count"] == 103,
    ["steam_name", "steam_supported_languages"]
].head(3)

for _, row in examples.iterrows():
    raw = str(row["steam_supported_languages"])

    print("\nGame:", row["steam_name"])
    print("String length:", len(raw))
    print("First 120 characters:", repr(raw[:120]))
    print("Last 120 characters:", repr(raw[-120:]))
    print("Ends with ]:", raw.rstrip().endswith("]"))

    try:
        parsed = ast.literal_eval(raw)
        print("Parsing successful:", len(parsed))
    except (ValueError, SyntaxError) as e:
        print("Parsing error:", str(e)[:200])

# Identify the single count mismatch
mismatch_mask = (
    df_clean["feature_language_count_parsed"].notna()
    & (
        df_clean["steam_language_count"]
        != df_clean["feature_language_count_parsed"]
    )
)

print("\nRecords with count mismatches:")

display(
    df_clean.loc[
        mismatch_mask,
        [
            "steam_name",
            "steam_language_count",
            "feature_language_count_parsed",
            "steam_supported_languages"
        ]
    ]
)



Game: Zup! X
String length: 1216
First 120 characters: "['English', 'French', 'Italian', 'German', 'Spanish - Spain', 'Arabic', 'Bulgarian', 'Portuguese - Brazil', 'Hungarian',"
Last 120 characters: "swana', 'Turkmen', 'Uzbek', 'Uyghur', 'Urdu', 'Filipino', 'Hausa', 'Hindi', 'Croatian', 'Cherokee', 'Scots', 'Estonian']"
Ends with ]: True
Parsing error: invalid syntax (<unknown>, line 1)

Game: Papetura
String length: 1216
First 120 characters: "['English', 'French', 'German', 'Spanish - Spain', 'Japanese', 'Korean', 'Polish', 'Portuguese - Brazil', 'Russian', 'Si"
Last 120 characters: "', 'Tigrinya', 'Tswana', 'Turkmen', 'Urdu', 'Uyghur', 'Uzbek', 'Valencian', 'Welsh', 'Wolof', 'Xhosa', 'Yoruba', 'Zulu']"
Ends with ]: True
Parsing error: invalid syntax (<unknown>, line 1)

Game: Last Room
String length: 1216
First 120 characters: "['English', 'French', 'Italian', 'German', 'Spanish - Spain', 'Arabic', 'Bulgarian', 'Portuguese - Brazil', 'Hungarian',"
Last 120 characters: "swana', 'Tur

,steam_name,steam_language_count,feature_language_count_parsed,steam_supported_languages


In [19]:
# Corrected language parsing (features.parse_supported_languages) was applied in the cell above.
# literal_eval is tried first; list strings it rejects, such as a language name containing an
# apostrophe, are split on their quotes instead.
print("Language feature reconstruction completed.")


Language feature reconstruction completed.


In [20]:

# Summarize parsing results
print("Parsing status:")
display(df_clean["language_parse_status"].value_counts())

print("\nCorrected language count summary:")
display(df_clean["feature_language_count"].describe())

# Check the previously problematic games
print("\nGames originally recorded with 103 languages:")

display(
    df_clean.loc[
        df_clean["steam_language_count"] == 103,
        [
            "steam_name",
            "steam_language_count",
            "feature_language_count",
            "language_parse_status"
        ]
    ].head(10)
)

# Identify remaining count disagreements
valid = df_clean["feature_language_count"].notna()

differences = df_clean.loc[
    valid & (
        df_clean["steam_language_count"]
        != df_clean["feature_language_count"]
    ),
    [
        "steam_name",
        "steam_language_count",
        "feature_language_count",
        "steam_supported_languages"
    ]
]

print("\nRemaining count disagreements:", len(differences))
display(differences.head(10))

assert df_clean["feature_language_count"].dropna().ge(0).all()

print("\nLanguage feature validation completed.")


Parsing status:


language_parse_status
standard    5273
fallback       8
Name: count, dtype: int64


Corrected language count summary:


count    5281.000000
mean        5.633971
std         6.385226
min         1.000000
25%         1.000000
50%         4.000000
75%         9.000000
max       102.000000
Name: feature_language_count, dtype: float64


Games originally recorded with 103 languages:


,steam_name,steam_language_count,feature_language_count,language_parse_status
643,Zup! X,103,102,fallback
1753,Papetura,103,102,fallback
2520,Last Room,103,102,fallback
2997,Shapik: The Moon Quest,103,102,fallback
3842,Winter,103,102,fallback
4590,Quadrata,103,102,fallback
4972,One Chance,103,102,fallback
5087,Tiny Onion Knight,103,102,fallback



Remaining count disagreements: 8


,steam_name,steam_language_count,feature_language_count,steam_supported_languages
643,Zup! X,103,102,"['English', 'French', 'Italian', 'German', 'Sp..."
1753,Papetura,103,102,"['English', 'French', 'German', 'Spanish - Spa..."
2520,Last Room,103,102,"['English', 'French', 'Italian', 'German', 'Sp..."
2997,Shapik: The Moon Quest,103,102,"['English', 'French', 'Italian', 'German', 'Sp..."
3842,Winter,103,102,"['English', 'German', 'Simplified Chinese', 'R..."
4590,Quadrata,103,102,"['English', 'Turkish', 'Italian', 'Polish', 'R..."
4972,One Chance,103,102,"['English', 'French', 'Italian', 'German', 'Sp..."
5087,Tiny Onion Knight,103,102,"['English', 'French', 'Italian', 'German', 'Sp..."



Language feature validation completed.


In [21]:

# Step 4: Finalize language feature engineering

# Columns added by features.add_language_features. Zero recorded languages are treated as
# missing information (flag_zero_languages), not as no language support, before the log transform.

print("Original vs corrected count differences:")
display(
    pd.crosstab(
        df_clean["language_parse_status"],
        df_clean["language_count_difference"]
    )
)




print("\nMissing language information:",
      df_clean["flag_zero_languages"].sum())

print("\nLog language count summary:")
display(
    df_clean["feature_log_language_count"].describe()
)

# Validation
assert df_clean["feature_language_count"].notna().all()

assert (
    df_clean["feature_log_language_count"]
    .dropna()
    .ge(0)
    .all()
)

print("\nLanguage feature engineering finalized.")


Original vs corrected count differences:


language_count_difference,0,1
language_parse_status,,
fallback,0,8
standard,5273,0



Missing language information: 0

Log language count summary:


count    5281.000000
mean        1.584440
std         0.768894
min         0.693147
25%         0.693147
50%         1.609438
75%         2.302585
max         4.634729
Name: feature_log_language_count, dtype: float64


Language feature engineering finalized.


### 1.5 Modelling Inclusion Thresholds
*Written by AI Assistant, revised for the current scope. To be reviewed by the group.*

Inclusion thresholds were set here, before any model was fitted, and are recorded in the feature specification.

The reception model requires at least 10 Steam reviews. This keeps 5,144 games. Free-to-play games are kept, and there is no HLTB requirement.

The completion model requires a story-driven game with a positive completion ratio (a SteamSpy playtime estimate), at least 50 Steam reviews and at least 5 HLTB main-story polls. This reduces the 5,281 games to **2,409 (45.62%)**.

Eligibility flags were created for both models. No games were removed from the cleaned dataset. All validation checks passed.

The cells below compare included and excluded games for each model. The restrictive completion-model criteria may introduce selection bias, which should be considered during subsequent analysis.

In [22]:

# Step 5: Define model inclusion thresholds

MIN_REVIEWS_RECEPTION = 10
MIN_REVIEWS_COMPLETION = 50
MIN_HLTB_POLLS = 5

# Reception and completion model eligibility
# The steps live in src/is630/features.py, so notebook 07 applies them to the test holdout unchanged.
from is630.features import add_eligibility_flags

df_clean = add_eligibility_flags(df_clean, MIN_REVIEWS_RECEPTION, MIN_REVIEWS_COMPLETION, MIN_HLTB_POLLS)

# Construct a cumulative selection funnel
completion_base = df_clean["flag_completion_eligible"]

funnel = pd.DataFrame({
    "Stage": [
        "Cleaned dataset",
        "Story-driven games",
        "Valid completion ratio",
        "At least 50 Steam reviews",
        "At least 5 HLTB polls"
    ],
    "Games": [
        len(df_clean),
        int(df_clean["flag_story_driven"].sum()),
        int(completion_base.sum()),
        int((
            completion_base
            & (df_clean["steam_review_count"] >= 50)
        ).sum()),
        int(df_clean["flag_completion_model"].sum())
    ]
})

funnel["Percentage of cleaned (%)"] = (
    funnel["Games"] / len(df_clean) * 100
).round(2)

display(funnel)

print(
    "Reception model sample:",
    f"{df_clean['flag_reception_model'].sum():,}"
)

print(
    "Completion model sample:",
    f"{df_clean['flag_completion_model'].sum():,}"
)

# Validation checks

assert (
    df_clean["flag_reception_model"]
    == df_clean["steam_review_count"].ge(MIN_REVIEWS_RECEPTION)
).all()

assert df_clean["flag_completion_model"].sum() <= completion_base.sum()
assert funnel["Games"].is_monotonic_decreasing

print("\nInclusion threshold checks passed.")


,Stage,Games,Percentage of cleaned (%)
0,Cleaned dataset,5281,100.00
1,Story-driven games,5281,100.00
2,Valid completion ratio,3605,68.26
3,At least 50 Steam reviews,3529,66.82
4,At least 5 HLTB polls,2409,45.62


Reception model sample: 5,144
Completion model sample: 2,409

Inclusion threshold checks passed.


In [23]:

# Step 5: Selection bias check — HLTB matched vs unmatched

selection_variables = [
    "steam_price",
    "steam_review_count",
    "steam_positive_rate",
    "steam_years_since_release"
]

hltb_selection_summary = (
    df_clean.assign(
        HLTB_Group=np.where(
            df_clean["matched_hltb"].fillna(False),
            "HLTB Matched",
            "HLTB Unmatched"
        )
    )
    .groupby("HLTB_Group")
    .agg(
        Games=("steam_appid", "size"),
        Median_Price=("steam_price", "median"),
        Median_Review_Count=("steam_review_count", "median"),
        Median_Positive_Rate=("steam_positive_rate", "median"),
        Median_Game_Age=("steam_years_since_release", "median")
    )
    .round(3)
    .reset_index()
)

display(hltb_selection_summary)

# Validate that every game is included
assert hltb_selection_summary["Games"].sum() == len(df_clean)

print("HLTB selection bias comparison checks passed.")


,HLTB_Group,Games,Median_Price,Median_Review_Count,Median_Positive_Rate,Median_Game_Age
0,HLTB Matched,5281,4.39,333.0,0.87,6.0


HLTB selection bias comparison checks passed.


In [24]:

# Step 5: Selection bias check — Completion model eligible vs excluded

completion_selection_summary = (
    df_clean.assign(
        Completion_Group=np.where(
            df_clean["flag_completion_model"],
            "Completion Eligible",
            "Completion Excluded"
        )
    )
    .groupby("Completion_Group")
    .agg(
        Games=("steam_appid", "size"),
        Median_Price=("steam_price", "median"),
        Median_Review_Count=("steam_review_count", "median"),
        Median_Positive_Rate=("steam_positive_rate", "median"),
        Median_Game_Age=("steam_years_since_release", "median")
    )
    .round(3)
    .reset_index()
)

display(completion_selection_summary)

# Validate group sizes
assert completion_selection_summary["Games"].sum() == len(df_clean)

assert (
    completion_selection_summary.loc[
        completion_selection_summary["Completion_Group"] == "Completion Eligible",
        "Games"
    ].iloc[0]
    == df_clean["flag_completion_model"].sum()
)

print("Completion selection bias comparison checks passed.")


,Completion_Group,Games,Median_Price,Median_Review_Count,Median_Positive_Rate,Median_Game_Age
0,Completion Eligible,2409,4.99,1648.0,0.877,7.0
1,Completion Excluded,2872,3.95,109.0,0.861,5.0


Completion selection bias comparison checks passed.


### 1.6 Training and Validation Split
*Written by AI Assistant, revised for the current scope. To be reviewed by the group.*

The 5,281 games were split into development-training (80%) and validation (20%) sets by stratified random sampling. Stratification keeps the share of reception-eligible and completion-eligible games about equal in both sets. The seed comes from `config.py`.

The development-training set has 4,224 games and the validation set has 1,057. Checks confirmed that every game is in exactly one set.

The language-count median fill uses the development-training median of 4 languages, applied to both sets. It replaced 0 values in each set, because no counts are missing. The missing-value indicator therefore has no variation and is dropped in section 1.7.

All imputation checks passed. The test holdout from notebook 00 is not loaded here.

In [25]:

# Step 6: Split training data into development and validation sets

from sklearn.model_selection import train_test_split

VALIDATION_SIZE = 0.20

# Validate eligibility relationship
assert not (
    df_clean["flag_completion_model"]
    & ~df_clean["flag_reception_model"]
).any()

# Stratify into three groups:
# 0 = Neither model
# 1 = Reception only
# 2 = Reception and completion
strata = (
    df_clean["flag_reception_model"].astype(int)
    + df_clean["flag_completion_model"].astype(int)
)

# Create development and validation datasets
df_train, df_val = train_test_split(
    df_clean,
    test_size=VALIDATION_SIZE,
    random_state=config.SEED,
    stratify=strata
)

df_train = df_train.copy()
df_val = df_val.copy()

# Summarize sample sizes
split_summary = pd.DataFrame({
    "Dataset": ["Development Train", "Validation"],
    "Total Games": [
        len(df_train),
        len(df_val)
    ],
    "Reception Eligible": [
        int(df_train["flag_reception_model"].sum()),
        int(df_val["flag_reception_model"].sum())
    ],
    "Completion Eligible": [
        int(df_train["flag_completion_model"].sum()),
        int(df_val["flag_completion_model"].sum())
    ]
})

display(split_summary)

# Validation checks
assert len(df_train) + len(df_val) == len(df_clean)

assert set(df_train["steam_appid"]).isdisjoint(
    set(df_val["steam_appid"])
)

assert (
    df_train["flag_reception_model"].sum()
    + df_val["flag_reception_model"].sum()
    == df_clean["flag_reception_model"].sum()
)

assert (
    df_train["flag_completion_model"].sum()
    + df_val["flag_completion_model"].sum()
    == df_clean["flag_completion_model"].sum()
)

print("\nTraining-validation split checks passed.")


,Dataset,Total Games,Reception Eligible,Completion Eligible
0,Development Train,4224,4115,1927
1,Validation,1057,1029,482



Training-validation split checks passed.


In [26]:

# Developer game count (training split only)
# Two steps use training data, so both use development training only (features.apply_training_stats):
# each developer's count of OTHER games in development training, so the feature means the same thing
# in both splits, and the median language count used to impute missing counts (checked below).
# The steps live in src/is630/features.py, so notebook 07 applies them to the test holdout unchanged.
from is630.features import apply_training_stats

df_train, df_val = apply_training_stats(df_train, df_train), apply_training_stats(df_val, df_train)

# Validation
assert df_train["feature_log_dev_game_count"].notna().all()
assert df_val["feature_log_dev_game_count"].notna().all()

assert df_train["feature_log_dev_game_count"].ge(0).all()
assert df_val["feature_log_dev_game_count"].ge(0).all()

print("Developer game count features created successfully.")


Developer game count features created successfully.


In [27]:

# Check website and support contact metadata

support_columns = [
    "steam_website",
    "steam_support_url",
    "steam_support_email"
]

print("Data types:")
print(df_clean[support_columns].dtypes)

print("\nExample values:")
display(df_clean[support_columns].head(10))

print("\nMissing values:")
print(df_clean[support_columns].isna().sum())


Data types:
steam_website          object
steam_support_url      object
steam_support_email    object
dtype: object

Example values:


,steam_website,steam_support_url,steam_support_email
0,https://www.gamesci.cn/wukong,None,cs@gamesci.com.cn
1,https://www.cyberpunk.net,http://en.cdprojektred.com/support,None
2,https://www.thewitcher.com/witcher3,http://en.cdprojektred.com/support,http://en.cdprojektred.com/contact-support/
3,https://baldursgate3.game,https://larian.com/support,None
4,https://www.callofduty.com/,https://support.activision.com/,None
5,http://www.thinkwithportals.com/,http://steamcommunity.com/app/620,None
6,None,https://www.bandainamcoent.com/support,None
7,http://www.fallout4.com,http://help.bethsoft.com/,None
8,https://www.hogwartslegacy.com/,https://portkeygamessupport.wbgames.com/hc,support@wbgames.com
9,http://elderscrolls.com/,http://support.bethsoft.com/,None



Missing values:
steam_website          1722
steam_support_url      2001
steam_support_email    1080
dtype: int64


In [28]:

# Step 6: Handle missing language values
# Use training-set statistics to prevent data leakage

# Calculate median from development-training data only
train_language_median = (
    df_train["feature_language_count_model"].median()
)

print(
    f"Training language-count median: "
    f"{train_language_median:.2f}"
)

# The imputed columns were added with the developer counts above (features.apply_training_stats)

# Summarize imputation results
imputation_summary = pd.DataFrame({
    "Dataset": ["Development Train", "Validation"],
    "Missing Before": [
        df_train["feature_language_count_model"].isna().sum(),
        df_val["feature_language_count_model"].isna().sum()
    ],
    "Missing After": [
        df_train["feature_language_count_imputed"].isna().sum(),
        df_val["feature_language_count_imputed"].isna().sum()
    ],
    "Imputed Median": [
        train_language_median,
        train_language_median
    ]
})

display(imputation_summary)

# Validation checks
assert df_train[
    "feature_language_count_imputed"
].notna().all()

assert df_val[
    "feature_language_count_imputed"
].notna().all()

assert df_train[
    "feature_log_language_count_imputed"
].notna().all()

assert df_val[
    "feature_log_language_count_imputed"
].notna().all()

print("\nLanguage imputation checks passed.")


Training language-count median: 4.00


,Dataset,Missing Before,Missing After,Imputed Median
0,Development Train,0,0,4.0
1,Validation,0,0,4.0



Language imputation checks passed.


In [29]:

# Step 6: Final validation of model-ready features

# Define common predictor variables

MODEL_FEATURES = [
    # Pricing
    "feature_log_price",
    "flag_free_to_play",

    # Game age and release window
    "steam_years_since_release",
    "flag_q4_release",

    # Platform support
    "steam_mac",
    "steam_linux",

    # Language and audio localization
    "feature_log_language_count_imputed",
    "feature_language_missing",
    "feature_log_audio_language_count",
    "flag_has_full_audio",

    # In-game and store features
    "feature_log_achievements",
    "flag_has_achievements",
    "flag_age_restricted",
    "cat_singleplayer",
    "cat_multiplayer",
    "cat_coop",
    "cat_controller",
    "cat_steam_cloud",
    "cat_trading_cards",
    "cat_vr",

    # Genres
    "genre_indie",
    "genre_casual",
    "genre_action",
    "genre_adventure",
    "genre_simulation",
    "genre_strategy",
    "genre_rpg",
    "genre_early_access",
    "flag_missing_genre",

    # Narrative
    "flag_story_driven",
    "flag_missing_tags",

    # Studio and publishing
    "flag_self_published",
    "feature_log_dev_game_count",

    # Storefront and support
    "flag_has_website",
    "flag_has_support_contact"
]

# The scope makes these constant: story-driven requires single-player and tags.
SCOPE_CONSTANT = ["cat_singleplayer", "flag_story_driven", "flag_missing_tags"]
MODEL_FEATURES = [f for f in MODEL_FEATURES if f not in SCOPE_CONSTANT]

# A predictor with no variation in a training sample cannot be estimated.
# Checked on the development-training rows each model fits on.
fit_samples = {
    "reception": df_train.loc[df_train["flag_reception_model"]],
    "completion": df_train.loc[df_train["flag_completion_model"]],
}
no_variation = [
    f for f in MODEL_FEATURES
    if any(sample[f].nunique() < 2 for sample in fit_samples.values())
]
MODEL_FEATURES = [f for f in MODEL_FEATURES if f not in no_variation]

print("Dropped, constant by scope:", SCOPE_CONSTANT)
print("Dropped, no variation in a training sample:", no_variation)
print("Model predictors:", len(MODEL_FEATURES))


# HLTB scope features for matched-subset analyses
HLTB_SCOPE_FEATURES = [
    "feature_log_main_story_hours",
    "flag_has_side_content"
]

validation_results = []

for name, dataset in [
    ("Development Train", df_train),
    ("Validation", df_val)
]:
    # Check all required predictors exist
    missing_cols = set(MODEL_FEATURES) - set(dataset.columns)
    assert not missing_cols, f"Missing columns: {missing_cols}"

    # Check numerical values
    X = dataset[MODEL_FEATURES].apply(
        pd.to_numeric, errors="coerce"
    )

    missing_values = int(X.isna().sum().sum())
    infinite_values = int(
        np.isinf(X.to_numpy(dtype=float)).sum()
    )

    # Check targets in their eligible samples
    reception = dataset.loc[
        dataset["flag_reception_model"],
        "steam_positive_rate"
    ]

    completion = dataset.loc[
        dataset["flag_completion_model"],
        "completion_ratio"
    ]

    assert reception.notna().all()
    assert reception.between(0, 1).all()

    assert completion.notna().all()
    assert completion.gt(0).all()

    assert missing_values == 0
    assert infinite_values == 0

    validation_results.append({
        "Dataset": name,
        "Games": len(dataset),
        "Predictors": len(MODEL_FEATURES),
        "Missing Predictor Values": missing_values,
        "Infinite Predictor Values": infinite_values,
        "Reception Eligible": len(reception),
        "Completion Eligible": len(completion)
    })

validation_summary = pd.DataFrame(validation_results)

display(validation_summary)

# Ensure no targets are included as predictors
TARGET_COLUMNS = [
    "steam_review_count",
    "steam_positive_rate",
    "completion_ratio"
]

assert not set(MODEL_FEATURES).intersection(TARGET_COLUMNS)

# Check that game IDs do not overlap
assert set(df_train["steam_appid"]).isdisjoint(
    set(df_val["steam_appid"])
)

print("\nAll model-ready feature checks passed.")


Dropped, constant by scope: ['cat_singleplayer', 'flag_story_driven', 'flag_missing_tags']
Dropped, no variation in a training sample: ['feature_language_missing']
Model predictors: 31


,Dataset,Games,Predictors,Missing Predictor Values,Infinite Predictor Values,Reception Eligible,Completion Eligible
0,Development Train,4224,31,0,0,4115,1927
1,Validation,1057,31,0,0,1029,482



All model-ready feature checks passed.


### 1.7 Model-Ready Data and Feature Specification
*Written by AI Assistant, revised for the current scope. To be reviewed by the group.*

Four predictors are dropped before saving. Three are constant by scope: `cat_singleplayer`, `flag_story_driven` and `flag_missing_tags`. One has no variation in the training sample: `feature_language_missing`. `feature_log_dev_game_count` counts each developer's other games, so it means the same thing in both sets.

Both sets contain the same 31 predictors, with no missing or infinite values. The development-training set has 4,224 games, of which 4,115 are reception-eligible and 1,927 completion-eligible. The validation set has 1,057 games, of which 1,029 are reception-eligible and 482 completion-eligible.

Both sets are saved as Parquet to `data/processed` for notebook 03 onwards. The feature specification (`02_feature_spec.json`) records the predictors, targets, inclusion thresholds and preprocessing parameters.

In [30]:

# Step 7: Save model-ready datasets

# Create output directory if necessary
config.PROCESSED.mkdir(parents=True, exist_ok=True)

# Define output paths
train_path = config.PROCESSED / "02_train.parquet"
val_path = config.PROCESSED / "02_validation.parquet"

# Remove duplicate column names before exporting
df_train = df_train.loc[:, ~df_train.columns.duplicated()].copy()
df_val = df_val.loc[:, ~df_val.columns.duplicated()].copy()

# Verify column names are unique
assert df_train.columns.is_unique
assert df_val.columns.is_unique

print("Duplicate column checks passed.")


# Save development-training and validation datasets
df_train.to_parquet(train_path, index=False)
df_val.to_parquet(val_path, index=False)

# Reload saved files to verify successful export
train_check = pd.read_parquet(train_path)
val_check = pd.read_parquet(val_path)

# Verify row counts and column consistency
assert len(train_check) == len(df_train)
assert len(val_check) == len(df_val)

assert set(train_check.columns) == set(val_check.columns)
assert set(MODEL_FEATURES).issubset(train_check.columns)

# Confirm no games overlap between datasets
assert set(train_check["steam_appid"]).isdisjoint(
    set(val_check["steam_appid"])
)

print("Model-ready datasets saved successfully.\n")

print("Development training:")
print(f"Path: {train_path}")
print(f"Shape: {train_check.shape}")

print("\nValidation:")
print(f"Path: {val_path}")
print(f"Shape: {val_check.shape}")

print("\nDataset export checks passed.")


Duplicate column checks passed.


Model-ready datasets saved successfully.

Development training:
Path: Z:\IS630\data\processed\02_train.parquet
Shape: (4224, 146)

Validation:
Path: Z:\IS630\data\processed\02_validation.parquet
Shape: (1057, 146)

Dataset export checks passed.


In [31]:

# Step 7: Save feature specification

import json

feature_spec = {
    "model_features": MODEL_FEATURES,
    "hltb_scope_features": HLTB_SCOPE_FEATURES,

    "targets": {
        "reception": "steam_positive_rate",
        "review_count": "steam_review_count",
        "completion": "completion_ratio"
    },

    "eligibility_flags": {
        "reception": "flag_reception_model",
        "completion": "flag_completion_model"
    },

    "inclusion_thresholds": {
        "min_reviews_reception": MIN_REVIEWS_RECEPTION,
        "min_reviews_completion": MIN_REVIEWS_COMPLETION,
        "min_hltb_polls": MIN_HLTB_POLLS,
        "story_driven_required_for_completion": True
    },

    "preprocessing": {
        "language_imputation": "median",
        "language_imputation_value": float(
            train_language_median
        ),
        "language_missing_indicator": "feature_language_missing",
        "language_predictor": "feature_log_language_count_imputed",
        "price_transformation": "log1p",
        "genre_encoding": "multi_label_binary",
        "genre_mapping": GENRE_MAP,
        "price_band_labels": PRICE_LABELS,
        "price_band_edges": [
            float(x) if np.isfinite(x) else "inf"
            for x in PRICE_BINS
        ],
        "game_age_reference_year": config.SNAPSHOT_YEAR
    },

    "validation_split": {
        "validation_fraction": VALIDATION_SIZE,
        "random_seed": config.SEED,
        "stratification": "reception_and_completion_eligibility"
    },

    "dataset_files": {
        "train": train_path.name,
        "validation": val_path.name
    }
}

# Save specification as JSON
spec_path = config.PROCESSED / "02_feature_spec.json"

with open(spec_path, "w", encoding="utf-8") as f:
    json.dump(feature_spec, f, indent=4, allow_nan=False)

# Reload to verify successful export
with open(spec_path, "r", encoding="utf-8") as f:
    saved_spec = json.load(f)

assert saved_spec["model_features"] == MODEL_FEATURES

print("Feature specification saved successfully.")
print(f"Path: {spec_path}")

print("\nNumber of predictors:",
      len(saved_spec["model_features"]))

print("Language imputation value:",
      saved_spec["preprocessing"]["language_imputation_value"])

print("\nFeature specification checks passed.")


Feature specification saved successfully.
Path: Z:\IS630\data\processed\02_feature_spec.json

Number of predictors: 31
Language imputation value: 4.0

Feature specification checks passed.
